# Notebook-Überblick: 01 Dataset Download

Dieses Notebook materialisiert externe Quellen als lokale Rohdaten unter code/input/ und stellt die Rohdaten für die lokale Verarbeitung bereit.

Datenfluss in diesem Schritt:
- Quelle -> Download/Übernahme -> lokale Dateien in code/input/
- Bereitstellung einer stabilen lokalen Rohdatenstruktur als Eingang für das zweite Notebook
- Die eigentliche Strukturierung und Qualitätsprüfung erfolgt erst im zweiten Notebook

Dateirollen in code/input/:
- GermanQuAD: queries.jsonl, corpus.jsonl, qrels.jsonl als lokale Roh-Inputs für den nächsten Schritt
- GerLeRB: topics.txt, corpus.trec.gz, qrels.txt als lokale Roh-Inputs für den nächsten Schritt
- Zenodo-Metadaten: zenodo_record_15745124.json dokumentiert die technische Herkunft und Dateistruktur des Downloads

Zielstruktur für nachgelagerte Verarbeitung:
- queries: Anfrage-ID und Anfrage-Text
- docs: Dokument-ID und Dokument-Text
- qrels: Zuordnung query_id -> doc_id mit Relevanzwert

# 01 Dataset Download (idempotent)

Dieses Notebook materialisiert Rohdaten lokal unter code/input/ und nutzt vorhandene Dateien idempotent wieder.

Lokale Zielstruktur:
- code/input/germanquad/queries.jsonl
- code/input/germanquad/corpus.jsonl
- code/input/germanquad/qrels.jsonl
- code/input/gerlerb/topics.txt
- code/input/gerlerb/corpus.trec.gz
- code/input/gerlerb/qrels.txt
- code/input/gerlerb/zenodo_record_15745124.json (optional technische Metadaten, wird später ignoriert)

In [1]:
from __future__ import annotations

import json
import os
import traceback
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd
import requests
from datasets import Dataset, DatasetDict, load_dataset

ROOT = Path.cwd()
if not (ROOT / "code").exists() and ROOT.name == "code":
    ROOT = ROOT.parent

INPUT_DIR = ROOT / "code" / "input"
GERMANQUAD_DIR = INPUT_DIR / "germanquad"
GERLERB_DIR = INPUT_DIR / "gerlerb"
HF_CACHE_DIR = INPUT_DIR / "hf_cache"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
GERMANQUAD_DIR.mkdir(parents=True, exist_ok=True)
GERLERB_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)

print(f"Working directory: {ROOT}")
print(f"Input directory: {INPUT_DIR}")

<PROJEKT>/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
<PROJEKT>/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working directory: <PROJEKT>
Input directory: <PROJEKT>/code/input


In [ ]:
def dataset_to_records(ds: Dataset) -> List[Dict[str, Any]]:
    return ds.to_pandas().to_dict(orient="records")


def write_jsonl(path: Path, records: List[Dict[str, Any]]) -> int:
    jsonl_split_threshold = 57_000
    path.parent.mkdir(parents=True, exist_ok=True)
    stale_parts = sorted(path.parent.glob(f"{path.stem}.part*{path.suffix}"))
    for stale_part in stale_parts:
        stale_part.unlink()

    written = 0
    handle = path.open("w", encoding="utf-8")
    try:
        for row in records:
            if written > 0 and written % jsonl_split_threshold == 0:
                handle.close()
                part_number = written // jsonl_split_threshold + 1
                part_path = path.with_name(f"{path.stem}.part{part_number}{path.suffix}")
                handle = part_path.open("w", encoding="utf-8")
            handle.write(json.dumps(row, ensure_ascii=True) + "\n")
            written += 1
    finally:
        handle.close()
    return written


def pick_first_split(data: DatasetDict) -> Dataset:
    if "test" in data:
        return data["test"]
    first_name = next(iter(data.keys()))
    return data[first_name]


def ensure_germanquad_file(target_path: Path, dataset_id: str, config: str | None) -> Dict[str, Any]:
    if target_path.exists() and target_path.stat().st_size > 0:
        return {"file": str(target_path.name), "status": "reused", "rows": None, "source": dataset_id, "config": config or ""}

    ds = load_dataset(dataset_id, config, cache_dir=str(HF_CACHE_DIR)) if config else load_dataset(dataset_id, cache_dir=str(HF_CACHE_DIR))
    selected = pick_first_split(ds) if isinstance(ds, DatasetDict) else ds
    written = write_jsonl(target_path, dataset_to_records(selected))
    return {"file": str(target_path.name), "status": "downloaded", "rows": written, "source": dataset_id, "config": config or ""}

In [3]:
status_rows: List[Dict[str, Any]] = []
errors: List[str] = []

germanquad_targets = [
    {"path": GERMANQUAD_DIR / "queries.jsonl", "dataset_id": "mteb/germanquad-retrieval", "config": "queries"},
    {"path": GERMANQUAD_DIR / "corpus.jsonl", "dataset_id": "mteb/germanquad-retrieval", "config": "corpus"},
    {"path": GERMANQUAD_DIR / "qrels.jsonl", "dataset_id": "mteb/germanquad-retrieval-qrels", "config": None},
]

for target in germanquad_targets:
    try:
        status_rows.append(ensure_germanquad_file(target["path"], target["dataset_id"], target["config"]))
    except Exception:
        errors.append(f"GermanQuAD fehlgeschlagen für {target['path'].name}: {traceback.format_exc(limit=1).strip()}")
        status_rows.append({"file": target["path"].name, "status": "failed", "rows": None, "source": target["dataset_id"], "config": target["config"] or ""})

display(pd.DataFrame(status_rows))
if errors:
    print("")
    print("Manuelle Eingabe für GermanQuAD erforderlich unter code/input/germanquad/:")
    print("- queries.jsonl (mind. query id + query text)")
    print("- corpus.jsonl (mind. doc id + text)")
    print("- qrels.jsonl (mind. query/doc id + relevance)")
    for err in errors:
        print(f"- {err}")

,file,status,rows,source,config
0,queries.jsonl,reused,None,mteb/germanquad-retrieval,queries
1,corpus.jsonl,reused,None,mteb/germanquad-retrieval,corpus
2,qrels.jsonl,reused,None,mteb/germanquad-retrieval-qrels,


In [4]:
required_gerlerb_files = ["topics.txt", "corpus.trec.gz", "qrels.txt"]
record_path = GERLERB_DIR / "zenodo_record_15745124.json"
gerlerb_status: Dict[str, Any] = {"dataset": "gerlerb", "required_files_present": True, "reused_files": [], "missing_files": [], "metadata_status": "skipped"}

for filename in required_gerlerb_files:
    fpath = GERLERB_DIR / filename
    if fpath.exists() and fpath.stat().st_size > 0:
        gerlerb_status["reused_files"].append(filename)
    else:
        gerlerb_status["required_files_present"] = False
        gerlerb_status["missing_files"].append(filename)

if gerlerb_status["required_files_present"]:
    if not record_path.exists():
        try:
            response = requests.get("https://zenodo.org/api/records/15745124", timeout=30)
            response.raise_for_status()
            record_path.write_text(json.dumps(response.json(), indent=2), encoding="utf-8")
            gerlerb_status["metadata_status"] = "downloaded"
        except Exception:
            gerlerb_status["metadata_status"] = "failed"
            gerlerb_status["metadata_error"] = traceback.format_exc(limit=1).strip()
    else:
        gerlerb_status["metadata_status"] = "reused"
else:
    print("GerLeRB Dateien fehlen lokal. Bitte manuell nach code/input/gerlerb/ legen:")
    for missing in gerlerb_status["missing_files"]:
        print(f"- {missing}")

display(pd.DataFrame([gerlerb_status]))

print("")
print("Lokale Dateiliste code/input/:")
for p in sorted(INPUT_DIR.rglob("*")):
    if p.is_file():
        print(f"- {p.relative_to(ROOT)}")

,dataset,required_files_present,reused_files,missing_files,metadata_status
0,gerlerb,True,"[topics.txt, corpus.trec.gz, qrels.txt]",[],reused



Lokale Dateiliste code/input/:
- code/input/gerlerb/corpus.trec.gz
- code/input/gerlerb/qrels.txt
- code/input/gerlerb/topics.txt
- code/input/gerlerb/zenodo_record_15745124.json
- code/input/germanquad/corpus.jsonl
- code/input/germanquad/qrels.jsonl
- code/input/germanquad/queries.jsonl
- code/input/hf_cache/hf_cache_mteb___germanquad-retrieval-qrels_default_0.0.0_ed232776ce5d736028c1da18e81ab7048969f369.lock
- code/input/hf_cache/hf_cache_mteb___germanquad-retrieval_corpus_0.0.0_9af36714ebd9bba5235c4b9862779adaddbfae52.lock
- code/input/hf_cache/hf_cache_mteb___germanquad-retrieval_queries_0.0.0_9af36714ebd9bba5235c4b9862779adaddbfae52.lock
- code/input/hf_cache/mteb___germanquad-retrieval/corpus/0.0.0/9af36714ebd9bba5235c4b9862779adaddbfae52/dataset_info.json
- code/input/hf_cache/mteb___germanquad-retrieval/corpus/0.0.0/9af36714ebd9bba5235c4b9862779adaddbfae52/germanquad-retrieval-corpus.arrow
- code/input/hf_cache/mteb___germanquad-retrieval/corpus/0.0.0/9af36714ebd9bba5235c4b98